In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import torch
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import sparse
from scipy.stats import kruskal, mannwhitneyu, spearmanr


def find_project_root() -> Path:
    """Resolve the repository root when running from either the root or analyses directory."""
    current_dir = Path.cwd().resolve()
    for candidate in (current_dir, current_dir.parent):
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from the project root or the analyses directory.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "03_spatial_analysis"
RESULTS_DIR = PROJECT_ROOT / "results" / "03_spatial_analysis"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from scvolt.model import SCVOLTModel
from scvolt import tools

CLASSIFICATION_COLUMN = "Classification"
CLASS_ORDER = ["Healthy", "Invasive", "Surrounding tumor", "Tumor"]
TUMOR_LIKE_CLASSES = ["Tumor", "Invasive"]
NON_TUMOR_CLASSES = ["Healthy", "Surrounding tumor"]
SENSITIVITY_TOP_PERCENTAGES = [10, 20, 25, 33, 50]

LAPATINIB_DRUG_NAME = "Lapatinib"
DASATINIB_DRUG_NAME = "Dasatinib"
RAW_SCVOLT_SCORE_COL = "scVOLT_score"
LAPATINIB_SCORE_COL = "scVOLT_sensitivity_score"
DASATINIB_SCORE_COL = "scVOLT_sensitivity_score_dasatinib"
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def result_path(filename: str) -> Path:
    """Return a path inside the spatial-analysis results directory."""
    return RESULTS_DIR / filename


def make_spatial_expression_frame(adata_obj) -> pd.DataFrame:
    """Convert an AnnData expression matrix into a spot-by-gene DataFrame."""
    if sparse.issparse(adata_obj.X):
        return pd.DataFrame.sparse.from_spmatrix(
            adata_obj.X,
            index=adata_obj.obs_names,
            columns=adata_obj.var_names,
        )
    return pd.DataFrame(
        adata_obj.X,
        index=adata_obj.obs_names,
        columns=adata_obj.var_names,
    )


def coverage_percent(mask, base) -> float:
    """Return the percentage of selected spots within a base mask."""
    denominator = np.sum(base)
    if denominator == 0:
        return np.nan
    return np.sum(mask & base) / denominator * 100


In [ ]:
model_latent_dim = 30
drug_name = LAPATINIB_DRUG_NAME
num_DRGs = 1200


In [ ]:
adata = sc.read_h5ad(DATA_DIR / "V19L29-097.h5ad")
spatial_expression_df = make_spatial_expression_frame(adata)


In [ ]:
tools.set_model_seed(43)
spatial_expression_data = {"st": spatial_expression_df}
lapatinib_model = SCVOLTModel(
    latent_dim=model_latent_dim,
    drug_name=drug_name,
    num_DRGs=num_DRGs,
)
lapatinib_encoded_data = lapatinib_model.encode(
    data=spatial_expression_data,
    device=TORCH_DEVICE,
)
lapatinib_predictions = lapatinib_model.predict(encoded_data=lapatinib_encoded_data)


In [ ]:
adata.obs[RAW_SCVOLT_SCORE_COL] = lapatinib_predictions["st_scores"]
adata.obs[LAPATINIB_SCORE_COL] = 1 - lapatinib_predictions["st_scores"]


In [ ]:
adata.obs[CLASSIFICATION_COLUMN]


In [ ]:
adata_use = adata[adata.obs[CLASSIFICATION_COLUMN].notna()].copy()
adata_use.obs[CLASSIFICATION_COLUMN] = adata_use.obs[CLASSIFICATION_COLUMN].astype("category")

palette = {
    "Healthy": "#67C1C9",
    "Tumor": "#D99A3D",
    "Invasive": "#B2182B",
    "Surrounding tumor": "#9E77ED",
}

order = ["Healthy", "Tumor", "Surrounding tumor", "Invasive"]

xy = adata_use.obsm["spatial"]
cls = adata_use.obs[CLASSIFICATION_COLUMN].astype(str)

fig, ax = plt.subplots(figsize=(6.2, 5.6))

for c in order:
    idx = cls == c
    if idx.sum() == 0:
        continue

    ax.scatter(
        xy[idx, 0],
        xy[idx, 1],
        s=20,
        c=palette[c],
        alpha=0.9,
        linewidths=0,
        rasterized=True
    )

ax.invert_yaxis()
ax.set_aspect("equal")

# Remove tick marks.
ax.set_xticks([])
ax.set_yticks([])

# Keep a thinner axis border.
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.0)
    spine.set_color("#444444")

# Legend.
handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markerfacecolor=palette[c],
        markeredgecolor="none",
        markersize=7,
        label=c
    )
    for c in order if c in cls.unique()
]

ax.legend(
    handles=handles,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=11,
    handletextpad=0.4
)

plt.subplots_adjust(right=0.78)

plt.savefig(result_path("spatial_classification_clean.pdf"),
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.02
)

plt.show()


In [ ]:

adata_use = adata[adata.obs[CLASSIFICATION_COLUMN].notna()].copy()

xy = adata_use.obsm["spatial"]
cls = adata_use.obs[CLASSIFICATION_COLUMN].astype(str)

# =========================
# Define tumor-like and non-tumor groups.
# =========================
tumor_like_classes = TUMOR_LIKE_CLASSES

group = np.where(
    cls.isin(tumor_like_classes),
    "Tumor-like",
    "Non-tumor"
)

adata_use.obs["Tumor_like_group"] = group

print(adata_use.obs["Tumor_like_group"].value_counts())

# =========================
# Color palette.
# =========================
palette = {
    "Non-tumor": "#D9D9D9",      # Light gray
    "Tumor-like": "#D98C8C",    # Light red
}

order = ["Non-tumor", "Tumor-like"]

# =========================
# Plot.
# =========================
fig, ax = plt.subplots(figsize=(6.2, 5.6))

for g in order:
    idx = adata_use.obs["Tumor_like_group"].values == g

    ax.scatter(
        xy[idx, 0],
        xy[idx, 1],
        s=20,
        c=palette[g],
        alpha=0.9,
        linewidths=0,
        rasterized=True
    )

ax.invert_yaxis()
ax.set_aspect("equal")

# Remove ticks and grid.
ax.set_xticks([])
ax.set_yticks([])
ax.grid(False)

# Keep axis border.
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.0)
    spine.set_color("#444444")

# Legend.
handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markerfacecolor=palette[g],
        markeredgecolor="none",
        markersize=7,
        label=g
    )
    for g in order
]

ax.legend(
    handles=handles,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=11,
    handletextpad=0.4
)

ax.set_title(
    "Tumor-like and non-tumor regions",
    fontsize=14,
    weight="bold"
)

plt.subplots_adjust(right=0.78)

plt.savefig(result_path("spatial_tumor_like_vs_nontumor_clean.pdf"),
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.02
)

plt.savefig(result_path("spatial_tumor_like_vs_nontumor_clean.png"),
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.02
)

plt.show()


In [ ]:
classification_score_df = adata.obs.loc[:, [CLASSIFICATION_COLUMN, LAPATINIB_SCORE_COL]].copy()

classification_summary_df = (
    classification_score_df.groupby(CLASSIFICATION_COLUMN)[LAPATINIB_SCORE_COL]
    .agg(
        n="count",
        median="median",
        q1=lambda values: values.quantile(0.25),
        q3=lambda values: values.quantile(0.75),
    )
)
classification_summary_df["IQR"] = classification_summary_df["q3"] - classification_summary_df["q1"]
classification_summary_df = classification_summary_df.round(4)

print(classification_summary_df)


In [ ]:
classification_series = adata.obs[CLASSIFICATION_COLUMN].astype(str)
score_groups = [
    adata.obs.loc[classification_series == class_label, RAW_SCVOLT_SCORE_COL].values
    for class_label in CLASS_ORDER
]

kruskal_result = kruskal(*score_groups)
print("Kruskal p (4 groups, excluding Artefact) =", kruskal_result)


In [ ]:
classification_series = adata.obs[CLASSIFICATION_COLUMN].astype(str)
tumor_like_mask = classification_series.isin(TUMOR_LIKE_CLASSES)
non_tumor_mask = classification_series.isin(NON_TUMOR_CLASSES)

score_key = LAPATINIB_SCORE_COL
tumor_like_scores = adata.obs.loc[tumor_like_mask, score_key].dropna().values
non_tumor_scores = adata.obs.loc[non_tumor_mask, score_key].dropna().values

mannwhitney_result = mannwhitneyu(tumor_like_scores, non_tumor_scores, alternative="two-sided")
mannwhitney_u = mannwhitney_result.statistic
mannwhitney_p_value = mannwhitney_result.pvalue

pairwise_differences = tumor_like_scores.reshape(-1, 1) - non_tumor_scores.reshape(1, -1)
greater_count = (pairwise_differences > 0).sum()
less_count = (pairwise_differences < 0).sum()
equal_count = (pairwise_differences == 0).sum()
cliffs_delta = (greater_count - less_count) / pairwise_differences.size

print("Comparison: Tumor-like vs Non-tumor")
print("scVOLT sensitivity score =", score_key)
print("n_high =", len(tumor_like_scores), "n_low =", len(non_tumor_scores))
print("median_high =", np.median(tumor_like_scores))
print("median_low =", np.median(non_tumor_scores))
print("median_diff =", np.median(tumor_like_scores) - np.median(non_tumor_scores))
print("mean_high =", np.mean(tumor_like_scores))
print("mean_low =", np.mean(non_tumor_scores))
print("mean_diff =", np.mean(tumor_like_scores) - np.mean(non_tumor_scores))
print("Mann-Whitney U =", mannwhitney_u)
print("p =", mannwhitney_p_value)
print("gt =", greater_count, "lt =", less_count, "eq =", equal_count, "pairs =", pairwise_differences.size)
print("Cliff's delta =", float(cliffs_delta))


In [ ]:

# =========================
# Plot dataframe
# =========================
plot_df = pd.DataFrame({
    "Group": (
        ["Tumor-like"] * len(tumor_like_scores) +
        ["Non-tumor"] * len(non_tumor_scores)
    ),
    "scVOLT_sensitivity_score": np.concatenate([tumor_like_scores, non_tumor_scores])
})

plot_df = plot_df.replace([np.inf, -np.inf], np.nan).dropna()

score_col = "scVOLT_sensitivity_score"
order = ["Tumor-like", "Non-tumor"]

palette = {
    "Tumor-like": "#B2182B",
    "Non-tumor": "#BDBDBD"
}

# =========================
# Style
# =========================
sns.set_theme(
    style="whitegrid",
    context="paper",
    rc={
        "axes.linewidth": 1.0,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    }
)

fig, ax = plt.subplots(figsize=(5.2, 6.0))

# =========================
# Boxplot
# =========================
sns.boxplot(
    data=plot_df,
    x="Group",
    y=score_col,
    order=order,
    palette=palette,
    width=0.42,
    showfliers=False,
    showcaps=False,
    linewidth=1.2,
    boxprops={"edgecolor": "black", "zorder": 3},
    whiskerprops={"color": "black", "linewidth": 1.1},
    medianprops={"color": "black", "linewidth": 1.8},
    ax=ax
)

# =========================
# Jitter points
# =========================
sns.stripplot(
    data=plot_df,
    x="Group",
    y=score_col,
    order=order,
    palette=palette,
    size=2.2,
    alpha=0.45,
    jitter=0.22,
    linewidth=0,
    ax=ax,
    zorder=2
)

# =========================
# Mean marker
# =========================
means = plot_df.groupby("Group")[score_col].mean()

for i, g in enumerate(order):
    ax.scatter(
        i,
        means[g],
        marker="D",
        s=42,
        facecolor="white",
        edgecolor="black",
        linewidth=1.0,
        zorder=10
    )

# =========================
# Significance annotation
# =========================
ymax = plot_df[score_col].max()
ymin = plot_df[score_col].min()
yrange = ymax - ymin
if yrange == 0:
    yrange = 1

y = ymax + 0.08 * yrange
h = 0.025 * yrange

ax.plot(
    [0, 0, 1, 1],
    [y, y + h, y + h, y],
    color="black",
    linewidth=1.0
)

ax.text(
    0.5,
    y + h + 0.015 * yrange,
    f"p = {mannwhitney_p_value:.2e}    Cliff's 鏈?= {cliffs_delta:.2f}",
    ha="center",
    va="bottom",
    fontsize=9
)

# =========================
# Labels
# =========================
ax.set_title("scVOLT sensitivity score", pad=14, fontsize=12)
ax.set_xlabel("")
ax.set_ylabel("scVOLT sensitivity score", fontsize=10)

ax.set_xticklabels([
    f"Tumor-like\nn = {len(tumor_like_scores)}",
    f"Non-tumor\nn = {len(non_tumor_scores)}"
])

# cleaner grid
ax.grid(axis="y", color="#E5E5E5", linewidth=0.8)
ax.grid(axis="x", visible=False)

sns.despine(ax=ax)

ax.set_ylim(ymin - 0.05 * yrange, ymax + 0.18 * yrange)

plt.tight_layout()

plt.savefig(result_path("compare_boxstrip.pdf"),
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.02
)

plt.show()


In [ ]:
# Receptor/adaptor layer.
erbb_receptor_signaling = ["EGFR","ERBB2","ERBB3","GRB2","SHC1","SOS1"]

erbb_pathway_activity = [
    "EGFR", "ERBB2", "ERBB3", "ERBB4",
    "EGF", "TGFA", "AREG", "EREG", "HBEGF",
    "NRG1", "NRG2",
    "GRB2", "SHC1", "SOS1",
    "KRAS", "BRAF", "RAF1",
    "MAP2K1", "MAP2K2", "MAPK1", "MAPK3",
    "PIK3CA", "PIK3R1", "AKT1", "AKT2",
    "MTOR", "RPS6KB1",
    "PLCG1", "SRC", "STAT3",
    "MYC", "JUN", "FOS", "CCND1"
]

stroma = [
    "COL1A1", "COL1A2", "DCN", "LUM", "FAP", "ACTA2"
]

immune = [
    "PTPRC", "CD3D", "CD3E", "MS4A1", "LST1", "LYZ"
]
# Keep only genes present in var_names.
erbb_receptor_signaling_use = [g for g in erbb_receptor_signaling if g in adata.var_names]
erbb_pathway_activity_use = [g for g in erbb_pathway_activity if g in adata.var_names]
stroma_use = [g for g in stroma if g in adata.var_names]
immune_use = [g for g in immune if g in adata.var_names]

print("erbb receptor signaling genes used:", erbb_receptor_signaling_use)
print("erbb pathway activity genes used:", erbb_pathway_activity_use)
print("stroma genes used:", stroma_use)
print("immune genes used:", immune_use)

sc.tl.score_genes(adata, erbb_receptor_signaling_use, score_name="erbb_receptor_signaling_score")
sc.tl.score_genes(adata, erbb_pathway_activity_use, score_name="erbb_pathway_activity_score")
sc.tl.score_genes(adata, stroma_use, score_name="stroma")
sc.tl.score_genes(adata, immune_use, score_name="immune")


In [ ]:
classification_series = adata.obs[CLASSIFICATION_COLUMN].astype(str)
tumor_like_mask = classification_series.isin(TUMOR_LIKE_CLASSES)
tumor_like_sensitivity = adata.obs.loc[tumor_like_mask, LAPATINIB_SCORE_COL].values

erbb_receptor_scores = adata.obs.loc[tumor_like_mask, "erbb_receptor_signaling_score"].values
erbb_pathway_scores = adata.obs.loc[tumor_like_mask, "erbb_pathway_activity_score"].values
stroma_scores = adata.obs.loc[tumor_like_mask, "stroma"].values
immune_scores = adata.obs.loc[tumor_like_mask, "immune"].values

erbb_receptor_corr, erbb_receptor_p = spearmanr(tumor_like_sensitivity, erbb_receptor_scores, nan_policy="omit")
erbb_pathway_corr, erbb_pathway_p = spearmanr(tumor_like_sensitivity, erbb_pathway_scores, nan_policy="omit")
stroma_corr, stroma_p = spearmanr(tumor_like_sensitivity, stroma_scores, nan_policy="omit")
immune_corr, immune_p = spearmanr(tumor_like_sensitivity, immune_scores, nan_policy="omit")

print("Tumor-like only (two invasive classes):")
print("scVOLT_score vs erbb_receptor_signaling_score   Spearman r =", erbb_receptor_corr, "p =", erbb_receptor_p)
print("scVOLT_score vs erbb_pathway_activity_score Spearman r =", erbb_pathway_corr, "p =", erbb_pathway_p)
print("Tumor-like only control signatures:")
print("scVOLT_score vs stroma     Spearman r =", stroma_corr, "p =", stroma_p)
print("scVOLT_score vs immune     Spearman r =", immune_corr, "p =", immune_p)


In [ ]:

# =========================
# Select only tumor-like spots.
# =========================
tumor_like_classes = ["Tumor", "Invasive"]

adata_use = adata[
    adata.obs["Classification"].notna()
    & adata.obs["Classification"].astype(str).isin(tumor_like_classes)
].copy()

x = adata_use.obs["scVOLT_sensitivity_score"].astype(float).values

pairs = [
    ("erbb_receptor_signaling_score", "ERBB receptor signaling", "#C44E52"),
    ("erbb_pathway_activity_score", "ERBB pathway activity", "#D98C3F"),
    ("stroma", "Stroma signature", "#4C78A8"),
    ("immune", "Immune signature", "#59A14F"),
]

# =========================
# Helper functions.
# =========================
def format_p(p):
    """Format a p-value for plot annotations."""
    if p == 0:
        return "p < 1e-300"
    elif p < 1e-3:
        return f"p = {p:.1e}"
    else:
        return f"p = {p:.3f}"

def binned_median(x, y, n_bins=12):
    """
    Bin x by quantiles and compute y median, Q1, and Q3 for each bin.
    This is more suitable than a standard regression line for nonlinear or non-normal data.
    """
    mask = np.isfinite(x) & np.isfinite(y)
    x_use = x[mask]
    y_use = y[mask]

    qs = np.linspace(0, 1, n_bins + 1)
    bins = np.quantile(x_use, qs)
    bins = np.unique(bins)

    centers = []
    medians = []
    q1s = []
    q3s = []

    for i in range(len(bins) - 1):
        if i == len(bins) - 2:
            idx = (x_use >= bins[i]) & (x_use <= bins[i + 1])
        else:
            idx = (x_use >= bins[i]) & (x_use < bins[i + 1])

        if idx.sum() < 5:
            continue

        centers.append(np.median(x_use[idx]))
        medians.append(np.median(y_use[idx]))
        q1s.append(np.quantile(y_use[idx], 0.25))
        q3s.append(np.quantile(y_use[idx], 0.75))

    return (
        np.array(centers),
        np.array(medians),
        np.array(q1s),
        np.array(q3s),
    )

# =========================
# Plot.
# =========================
fig, axes = plt.subplots(2, 2, figsize=(8.5, 7.2))
axes = axes.ravel()

for ax, (col, label, color) in zip(axes, pairs):
    y = adata_use.obs[col].astype(float).values

    mask = np.isfinite(x) & np.isfinite(y)
    x_use = x[mask]
    y_use = y[mask]

    r, p = spearmanr(x_use, y_use)

    # Scatter points.
    ax.scatter(
        x_use,
        y_use,
        s=12,
        color=color,
        alpha=0.35,
        linewidths=0,
        rasterized=True
    )

    # Quantile trend line.
    bx, by, bq1, bq3 = binned_median(x_use, y_use, n_bins=12)

    ax.plot(
        bx,
        by,
        color=color,
        linewidth=2.5
    )

    ax.fill_between(
        bx,
        bq1,
        bq3,
        color=color,
        alpha=0.18,
        linewidth=0
    )

    # Correlation annotation.
    ax.text(
        0.05,
        0.95,
        f"Spearman r = {r:.3f}\n{format_p(p)}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="white",
            edgecolor="#999999",
            alpha=0.9
        )
    )

    ax.set_title(label, fontsize=12, weight="bold")
    ax.set_xlabel("scVOLT sensitivity score", fontsize=11)
    ax.set_ylabel(label, fontsize=11)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)
    ax.tick_params(axis="both", labelsize=9)

fig.suptitle(
    "Association between predicted Lapatinib sensitivity and pathway signatures\n"
    "in tumor-like regions",
    fontsize=14,
    weight="bold",
    y=1.03
)

plt.tight_layout()

plt.savefig(result_path("correlation_scVOLT_sensitivity_pathway_scores_tumor_like.pdf"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(result_path("correlation_scVOLT_sensitivity_pathway_scores_tumor_like.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# =========================
# 1. Groups and colors.
# =========================
class_order = [
    "Healthy",
    "Invasive",
    "Surrounding tumor",
    "Tumor"
]

palette = {
    "Healthy": "#2ECC71",
    "Invasive": "#FF2D55",
    "Surrounding tumor": "#FFD60A",
    "Tumor": "#00D1FF",
}

# Detect the classification column automatically.
if "Classification" in adata.obs.columns:
    class_col = "Classification"
elif "Classifications" in adata.obs.columns:
    class_col = "Classifications"
else:
    raise ValueError("Could not find a Classification or Classifications column in adata.obs")

# Keep only the four classes to display.
adata_use = adata[
    adata.obs[class_col].astype(str).isin(class_order)
].copy()

adata_use.obs[class_col] = pd.Categorical(
    adata_use.obs[class_col].astype(str),
    categories=class_order,
    ordered=True
)

# =========================
# 2. Scores to plot.
# =========================
keys = [
    LAPATINIB_SCORE_COL,
    "erbb_receptor_signaling_score",
    "erbb_pathway_activity_score",
]

titles = {
    LAPATINIB_SCORE_COL: "scVOLT sensitivity score",
    "erbb_receptor_signaling_score": "ERBB receptor signaling",
    "erbb_pathway_activity_score": "ERBB pathway activity",
}

# Check that required columns exist.
missing = [k for k in keys if k not in adata_use.obs.columns]
if len(missing) > 0:
    raise ValueError(
        f"These scores are not in adata.obs.columns: {missing}\n"
        f"Currently available columns include:\n{list(adata.obs.columns)}"
    )

# Ensure scores are numeric.
for k in keys:
    adata_use.obs[k] = pd.to_numeric(adata_use.obs[k], errors="coerce")

# =========================
# 3. Draw violin plots.
# =========================
axes = sc.pl.violin(
    adata_use,
    keys=keys,
    groupby=class_col,
    order=class_order,
    stripplot=False,
    jitter=False,
    show=False,
    palette=palette,
    multi_panel=True,
)

axes = np.ravel(axes)

for ax, key in zip(axes, keys):
    ax.set_title(titles[key], fontsize=12)
    ax.set_xlabel("")
    
    # Hide long x-axis category names if needed.
    ax.set_xticklabels([])
    ax.tick_params(bottom=False)

    # Style violin plots.
    for pc in ax.collections:
        try:
            pc.set_alpha(1.0)
            pc.set_edgecolor("none")
        except Exception:
            pass

    # Use an independent y-axis range for each metric.
    vals = adata_use.obs[key].dropna().values
    if len(vals) > 0:
        ymin, ymax = np.percentile(vals, [1, 99])
        pad = (ymax - ymin) * 0.25 if ymax > ymin else 0.1
        ax.set_ylim(ymin - pad, ymax + pad)

# Uncomment to fix scVOLT to the -1 to 1 range.
# axes[0].set_ylim(-1, 1)

# =========================
# 4. Shared legend.
# =========================
fig = axes[0].figure

handles = [
    Patch(facecolor=palette[k], edgecolor="none", label=k)
    for k in class_order
]

axes[-1].legend(
    handles=handles,
    title="Classification",
    loc="upper left",
    bbox_to_anchor=(1.02, 1.0),
    frameon=False,
    fontsize=9,
    title_fontsize=10
)

fig.set_size_inches(9.5, 3.2)
fig.tight_layout(rect=[0, 0, 0.82, 1])

plt.savefig(result_path("CID4290_erbb_classification_violin.pdf"), bbox_inches="tight")
plt.savefig(result_path("CID4290_erbb_classification_violin.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

# =========================
# Data.
# =========================
adata_use = adata.copy()
xy = adata_use.obsm["spatial"]

plot_info = [
    (
        "scVOLT_sensitivity_score",
        "Lapatinib sensitivity",
        0.02,
        0.10
    ),
    (
        "erbb_receptor_signaling_score",
        "ERBB receptor signaling",
        -0.8,
        1.5
    ),
    (
        "erbb_pathway_activity_score",
        "ERBB downstream activity",
        -0.3,
        2
    )
]

# =========================
# Plot.
# =========================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (col, title, vmin, vmax) in zip(axes, plot_info):

    score = adata_use.obs[col].astype(float).values

    sc_plot = ax.scatter(
        xy[:, 0],
        xy[:, 1],
        c=score,
        s=12,
        cmap="RdBu_r",
        vmin=vmin,
        vmax=vmax,
        linewidths=0,
        rasterized=True
    )

    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=13)

    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)

    cbar = plt.colorbar(
        sc_plot,
        ax=ax,
        fraction=0.046,
        pad=0.04
    )
    cbar.set_label("Score", fontsize=10)

plt.tight_layout()

plt.savefig(result_path("CID4290_scVOLT_validation_RdBu.pdf"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(result_path("CID4290_scVOLT_validation_RdBu.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
overlay_adata = adata[
    adata.obs[CLASSIFICATION_COLUMN].notna()
    & adata.obs[CLASSIFICATION_COLUMN].astype(str).isin(CLASS_ORDER)
    & adata.obs[LAPATINIB_SCORE_COL].notna()
].copy()

classification_series = overlay_adata.obs[CLASSIFICATION_COLUMN].astype(str)
tumor_like_mask = classification_series.isin(TUMOR_LIKE_CLASSES).values
non_tumor_mask = classification_series.isin(NON_TUMOR_CLASSES).values

spatial_coords = overlay_adata.obsm["spatial"]
lapatinib_scores = overlay_adata.obs[LAPATINIB_SCORE_COL].astype(float).values
lapatinib_top_percent = 25
lapatinib_threshold = np.nanpercentile(lapatinib_scores, 100 - lapatinib_top_percent)
lapatinib_sensitive = lapatinib_scores >= lapatinib_threshold

fig, ax = plt.subplots(figsize=(4.5, 4.5))

ax.scatter(
    spatial_coords[non_tumor_mask, 0],
    spatial_coords[non_tumor_mask, 1],
    s=18,
    c="#D9D9D9",
    edgecolors="white",
    linewidths=0.3,
)

ax.scatter(
    spatial_coords[tumor_like_mask, 0],
    spatial_coords[tumor_like_mask, 1],
    s=18,
    c="#D98C8C",
    edgecolors="white",
    linewidths=0.3,
)

ax.scatter(
    spatial_coords[lapatinib_sensitive, 0],
    spatial_coords[lapatinib_sensitive, 1],
    s=28,
    c="#3366FF",
    edgecolors="white",
    linewidths=0.4,
)

ax.set_title("Spatial overlay", fontsize=13, fontweight="bold")
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")

handles = [
    Patch(facecolor="#D98C8C", edgecolor="none", label="Tumor-like"),
    Patch(facecolor="#D9D9D9", edgecolor="none", label="Non-tumor"),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="#3366FF",
        markeredgecolor="white",
        markersize=7,
        label="Sensitive spots",
    ),
]

ax.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.25),
    frameon=False,
    fontsize=9,
    ncol=1,
)
ax.invert_yaxis()
plt.tight_layout()

plt.savefig(result_path("CID4290_sensitivity_spatial_overlay.pdf"), bbox_inches="tight")
plt.savefig(result_path("CID4290_sensitivity_spatial_overlay.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

tumor_classes = TUMOR_LIKE_CLASSES
adata_tumor = adata[adata.obs[CLASSIFICATION_COLUMN].isin(tumor_classes)].copy()
score_col = LAPATINIB_SCORE_COL
median_score = adata_tumor.obs[score_col].median()

adata_tumor.obs["drug_response_group"] = np.where(
    adata_tumor.obs[score_col] >= median_score,
    "Sensitive",
    "Resistant",
)
adata_tumor.obs["drug_response_group"] = pd.Categorical(
    adata_tumor.obs["drug_response_group"],
    categories=["Resistant", "Sensitive"],
)
adata_tumor.write_h5ad(result_path("adata_tumor_sensitive_resistant.h5ad"))


In [ ]:
sum([1 for x in adata_tumor.obs['drug_response_group'] if x == 'Sensitive'])


In [ ]:
adata_tumor = sc.read_h5ad(result_path("adata_tumor_sensitive_resistant.h5ad"))

sc.pp.filter_genes(adata_tumor, min_cells=10)

sc.pp.normalize_total(adata_tumor, target_sum=1e4)
sc.pp.log1p(adata_tumor)

sc.tl.rank_genes_groups(
    adata_tumor,
    groupby="drug_response_group",
    groups=["Resistant"],
    reference="Sensitive",
    method="wilcoxon"
)

deg = sc.get.rank_genes_groups_df(
    adata_tumor,
    group="Resistant"
).sort_values("pvals_adj")

deg.to_csv(result_path("02_DEG_Resistant_vs_Sensitive.csv"), index=False)

resistant_up = deg[
    (deg["logfoldchanges"] > 0.25) &
    (deg["pvals_adj"] < 0.05)
].copy()

resistant_up.to_csv(result_path("02_Resistant_up_genes.csv"), index=False)

print("Resistant up genes:", resistant_up.shape[0])
display(resistant_up.head(20))

deg["minus_log10_padj"] = -np.log10(deg["pvals_adj"] + 1e-300)

fig, ax = plt.subplots(figsize=(5, 4))

ax.scatter(
    deg["logfoldchanges"],
    deg["minus_log10_padj"],
    s=8,
    c="#D0D0D0",
    alpha=0.6,
    linewidths=0,
    rasterized=True
)

sig_up = (deg["logfoldchanges"] > 0.25) & (deg["pvals_adj"] < 0.05)

ax.scatter(
    deg.loc[sig_up, "logfoldchanges"],
    deg.loc[sig_up, "minus_log10_padj"],
    s=10,
    c="#C8674F",
    alpha=0.8,
    linewidths=0,
    rasterized=True
)

ax.axvline(0.25, linestyle="--", color="black", linewidth=0.8)
ax.axhline(-np.log10(0.05), linestyle="--", color="black", linewidth=0.8)

ax.set_xlabel("log fold change")
ax.set_ylabel("-log10 adjusted p-value")
ax.set_title("Resistant vs Sensitive")

plt.savefig(result_path("02_DEG_volcano.pdf"), dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
import gseapy as gp

resistant_up = pd.read_csv(result_path("02_Resistant_up_genes.csv"))

gene_list = resistant_up["names"].dropna().astype(str).unique().tolist()

print("Number of genes:", len(gene_list))
print(gene_list[:20])

enr = gp.enrichr(
    gene_list=gene_list,
    gene_sets=[
        "GO_Biological_Process_2023",
        "Reactome_2022",
        "KEGG_2021_Human",
        "MSigDB_Hallmark_2020"
    ],
    organism="human",
    outdir=str(result_path("03_enrichment_resistant_up")),
    cutoff=0.5
)

enrich_df = enr.results.copy()
enrich_df = enrich_df.sort_values("Adjusted P-value")

enrich_df.to_csv(result_path("03_Resistant_up_pathway_enrichment.csv"), index=False)

display(enrich_df.head(20))

plot_df = enrich_df.head(15).copy()
plot_df["minus_log10_padj"] = -np.log10(plot_df["Adjusted P-value"] + 1e-300)
plot_df = plot_df.sort_values("minus_log10_padj")

fig, ax = plt.subplots(figsize=(6, 5))

ax.barh(
    plot_df["Term"],
    plot_df["minus_log10_padj"],
    color="#C8674F"
)

ax.set_xlabel("-log10 adjusted p-value")
ax.set_ylabel("")
ax.set_title("Resistant-up pathway enrichment")

plt.tight_layout()
plt.savefig(result_path("03_resistant_pathway_enrichment.pdf"), dpi=600, bbox_inches="tight")
plt.show()


In [ ]:

# =========================
# Load enrichment results.
# =========================
enrich_df = pd.read_csv(result_path("03_Resistant_up_pathway_enrichment.csv"))

# =========================
# Prepare data.
# =========================
plot_df = enrich_df.copy()
plot_df = plot_df.sort_values("Adjusted P-value").head(20).copy()

plot_df["minus_log10_padj"] = -np.log10(plot_df["Adjusted P-value"] + 1e-300)

plot_df["Overlap_count"] = (
    plot_df["Overlap"]
    .astype(str)
    .str.split("/")
    .str[0]
    .astype(float)
)

plot_df["Term_short"] = (
    plot_df["Term"]
    .str.replace(r"\s*\(GO:\d+\)", "", regex=True)
    .str.replace("_", " ")
)

plot_df = plot_df.sort_values("minus_log10_padj")

# =========================
# Plot.
# =========================
fig, ax = plt.subplots(figsize=(12, 8.5))

gene_set_colors = {
    "GO_Biological_Process_2023": "#4C78A8",
    "Reactome_2022": "#F58518",
    "KEGG_2021_Human": "#54A24B",
    "MSigDB_Hallmark_2020": "#B279A2",
}

plot_df["color"] = plot_df["Gene_set"].map(gene_set_colors).fillna("#999999")

ax.scatter(
    plot_df["minus_log10_padj"],
    plot_df["Term_short"],
    s=plot_df["Overlap_count"] * 18,
    c=plot_df["color"],
    edgecolor="black",
    linewidth=0.5,
    alpha=0.9
)

ax.set_xlabel("-log10 adjusted p-value", fontsize=12)
ax.set_ylabel("")
ax.set_title("Pathway enrichment of resistant-up genes", fontsize=14, weight="bold")

xmin = plot_df["minus_log10_padj"].min()
xmax = plot_df["minus_log10_padj"].max()
xrange = xmax - xmin

# Leave extra right-side space and place the legend inside the axis.
ax.set_xlim(
    max(0, xmin - xrange * 0.25),
    xmax + xrange * 1.3
)

# =========================
# Gene_set legend.
# =========================
gene_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markerfacecolor=color,
        markeredgecolor="black",
        label=name.replace("_", " ")
    )
    for name, color in gene_set_colors.items()
]

legend1 = ax.legend(
    handles=gene_handles,
    title="Gene set",
    frameon=False,
    loc="upper right",
    fontsize=9,
    title_fontsize=10
)

ax.add_artist(legend1)

# =========================
# Overlap legend.
# =========================
size_levels = [5, 10, 20]
size_handles = []

for s in size_levels:
    ms = np.sqrt(s * 18)
    size_handles.append(
        Line2D(
            [0], [0],
            marker="o",
            linestyle="",
            markerfacecolor="gray",
            markeredgecolor="black",
            markersize=ms,
            label=f"{s} genes"
        )
    )

legend2 = ax.legend(
    handles=size_handles,
    title="Overlap",
    frameon=False,
    loc="center right",
    fontsize=9,
    title_fontsize=10,
    labelspacing=1.5,
    handletextpad=1.2
)

ax.add_artist(legend2)

ax.tick_params(axis="y", labelsize=10, pad=6)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.savefig(result_path("03_resistant_pathway_enrichment_bubble.pdf"), dpi=600, bbox_inches="tight")
plt.savefig(result_path("03_resistant_pathway_enrichment_bubble.png"), dpi=300, bbox_inches="tight")

plt.show()


In [ ]:
model_latent_dim = 30
drug_name = DASATINIB_DRUG_NAME
num_DRGs = 1200


In [ ]:
spatial_expression_df = make_spatial_expression_frame(adata)


In [ ]:
tools.set_model_seed(43)
spatial_expression_data = {"st": spatial_expression_df}
dasatinib_model = SCVOLTModel(
    latent_dim=model_latent_dim,
    drug_name=drug_name,
    num_DRGs=num_DRGs,
)
dasatinib_encoded_data = dasatinib_model.encode(
    data=spatial_expression_data,
    device=TORCH_DEVICE,
)
dasatinib_predictions = dasatinib_model.predict(encoded_data=dasatinib_encoded_data)


In [ ]:
adata.obs[DASATINIB_SCORE_COL] = 1 - dasatinib_predictions["st_scores"]


In [ ]:

drug1_col = LAPATINIB_SCORE_COL
drug2_col = DASATINIB_SCORE_COL

adata_use = adata[adata.obs[CLASSIFICATION_COLUMN].notna()].copy()

tumor_classes = ["Tumor", "Invasive", "Surrounding tumor", "Healthy"]
adata_use = adata_use[adata_use.obs[CLASSIFICATION_COLUMN].isin(tumor_classes)].copy()

xy = adata_use.obsm["spatial"]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, col, title in zip(
    axes,
    [drug1_col, drug2_col],
    [f"{LAPATINIB_DRUG_NAME} sensitivity", f"{DASATINIB_DRUG_NAME} sensitivity"]
):
    score = adata_use.obs[col].astype(float).values

    # Use quantiles for color scaling to reduce the effect of outliers.
    vmin = np.nanpercentile(score, 2)
    vmax = np.nanpercentile(score, 98)

    sc = ax.scatter(
        xy[:, 0],
        xy[:, 1],
        c=score,
        s=10,
        cmap="RdBu_r",
        vmin=vmin,
        vmax=vmax,
        linewidths=0,
        rasterized=True
    )

    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)

    cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Sensitivity score", fontsize=10)

plt.tight_layout()
plt.savefig(result_path("two_drug_sensitivity_spatial.pdf"), dpi=600, bbox_inches="tight")
plt.show()


In [ ]:

# =========================
# Drug columns
# =========================
drugA_col = LAPATINIB_SCORE_COL
drugB_col = DASATINIB_SCORE_COL

drugA_name = LAPATINIB_DRUG_NAME
drugB_name = DASATINIB_DRUG_NAME
# =========================
# Data preparation
# =========================
adata_use = adata[
    adata.obs[CLASSIFICATION_COLUMN].notna()
    & adata.obs[drugA_col].notna()
    & adata.obs[drugB_col].notna()
].copy()

xy = adata_use.obsm["spatial"]

sA = adata_use.obs[drugA_col].astype(float).values
sB = adata_use.obs[drugB_col].astype(float).values

print("Number of spots used:", adata_use.n_obs)

# =========================
# Thresholds to compare
# =========================
top_percents = SENSITIVITY_TOP_PERCENTAGES
# top_percents = [25]

colors = {
    "neither": "#BDBDBD",
    "A_only": "#F46D43",
    "B_only": "#7FB069",
    "both": "#7A3EB1",
}

# =========================
# Multi-panel figure
# =========================
n = len(top_percents)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), constrained_layout=True)

if n == 1:
    axes = [axes]

summary_rows = []

for ax, top_percent in zip(axes, top_percents):

    # -------------------------
    # threshold
    # -------------------------
    tA = np.nanpercentile(sA, 100 - top_percent)
    tB = np.nanpercentile(sB, 100 - top_percent)

    senA = sA >= tA
    senB = sB >= tB

    both = senA & senB
    A_only = senA & (~senB)
    B_only = senB & (~senA)
    neither = (~senA) & (~senB)

    # -------------------------
    # print summary
    # -------------------------
    total = len(sA)

    summary_rows.append({
        "top_percent": top_percent,
        "Lapatinib_threshold": tA,
        "Dasatinib_threshold": tB,
        "Lapatinib_only_n": A_only.sum(),
        "Dasatinib_only_n": B_only.sum(),
        "Sensitive_to_both_n": both.sum(),
        "Insensitive_to_both_n": neither.sum(),
        "Lapatinib_only_percent": A_only.sum() / total * 100,
        "Dasatinib_only_percent": B_only.sum() / total * 100,
        "Sensitive_to_both_percent": both.sum() / total * 100,
        "Insensitive_to_both_percent": neither.sum() / total * 100,
    })

    # -------------------------
    # plot
    # -------------------------
    ax.scatter(
        xy[neither, 0], xy[neither, 1],
        s=8, c=colors["neither"],
        alpha=0.65, linewidths=0, rasterized=True
    )

    ax.scatter(
        xy[A_only, 0], xy[A_only, 1],
        s=8, c=colors["A_only"],
        alpha=0.90, linewidths=0, rasterized=True
    )

    ax.scatter(
        xy[B_only, 0], xy[B_only, 1],
        s=8, c=colors["B_only"],
        alpha=0.90, linewidths=0, rasterized=True
    )

    ax.scatter(
        xy[both, 0], xy[both, 1],
        s=8, c=colors["both"],
        alpha=0.95, linewidths=0, rasterized=True
    )

    ax.set_title(
        f"Top {top_percent}% sensitive regions\n"
        f"Both: {both.sum()} spots",
        fontsize=12,
        weight="bold"
    )

    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)

# =========================
# Shared legend
# =========================
handles = [
    Line2D(
        [0], [0],
        marker="o", linestyle="",
        color=colors["both"],
        label=f"Sensitive to both",
        markersize=7
    ),
    Line2D(
        [0], [0],
        marker="o", linestyle="",
        color=colors["A_only"],
        label=f"{drugA_name} only",
        markersize=7
    ),
    Line2D(
        [0], [0],
        marker="o", linestyle="",
        color=colors["B_only"],
        label=f"{drugB_name} only",
        markersize=7
    ),
    Line2D(
        [0], [0],
        marker="o", linestyle="",
        color=colors["neither"],
        label="Insensitive to both",
        markersize=7
    ),
]

fig.legend(
    handles=handles,
    frameon=False,
    fontsize=11,
    loc="lower center",
    ncol=4,
    bbox_to_anchor=(0.5, -0.04)
)

fig.suptitle(
    f"Overlap of predicted sensitive regions: {drugA_name} vs {drugB_name}",
    fontsize=16,
    weight="bold",
    y=1.04
)

plt.savefig(result_path("03_overlap_sensitive_regions_multiple_thresholds.pdf"),
    dpi=600,
    bbox_inches="tight"
)
plt.savefig(result_path("03_overlap_sensitive_regions_multiple_thresholds.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# =========================
# Summary table
# =========================

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(result_path("03_overlap_sensitive_regions_threshold_summary.csv"), index=False)

print(summary_df)


In [ ]:
# =========================
# Basic settings.
# =========================
drugA_col = LAPATINIB_SCORE_COL
drugB_col = DASATINIB_SCORE_COL

drugA_name = LAPATINIB_DRUG_NAME
drugB_name = DASATINIB_DRUG_NAME
top_percents = SENSITIVITY_TOP_PERCENTAGES
# top_percents = [25]

# =========================
# Data preparation.
# =========================
adata_use = adata[
    adata.obs[CLASSIFICATION_COLUMN].notna()
    & adata.obs[drugA_col].notna()
    & adata.obs[drugB_col].notna()
].copy()

lapatinib_scores = adata_use.obs[drugA_col].astype(float).values
dasatinib_scores = adata_use.obs[drugB_col].astype(float).values
classification_values = adata_use.obs[CLASSIFICATION_COLUMN].astype(str).values

print("Number of spots used:", adata_use.n_obs)
print("Classification counts:")
print(adata_use.obs[CLASSIFICATION_COLUMN].value_counts())

# =========================
# Tumor-region definition.
# =========================
# Main analysis should use only Tumor + Invasive.
tumor_like = np.isin(classification_values, TUMOR_LIKE_CLASSES)

# Non-tumor regions are all classified spots outside Tumor/Invasive.
non_tumor = ~tumor_like

print("Tumor-like spots:", tumor_like.sum())
print("Non-tumor spots:", non_tumor.sum())


# =========================
# Multi-threshold calculation.
# =========================
summary_rows = []

for top_percent in top_percents:
    lapatinib_threshold = np.nanpercentile(lapatinib_scores, 100 - top_percent)
    dasatinib_threshold = np.nanpercentile(dasatinib_scores, 100 - top_percent)

    lapatinib_sensitive = lapatinib_scores >= lapatinib_threshold
    dasatinib_sensitive = dasatinib_scores >= dasatinib_threshold

    combined_sensitive = lapatinib_sensitive | dasatinib_sensitive
    added_by_dasatinib = (~lapatinib_sensitive) & dasatinib_sensitive

    metrics = {
        f"{drugA_name} coverage in tumor-like": coverage_percent(lapatinib_sensitive, tumor_like),
        f"{drugB_name} coverage in tumor-like": coverage_percent(dasatinib_sensitive, tumor_like),
        "Combined coverage in tumor-like": coverage_percent(combined_sensitive, tumor_like),
        f"New tumor-like coverage by {drugB_name}": coverage_percent(added_by_dasatinib, tumor_like),
        f"Non-tumor covered by {drugB_name}": coverage_percent(dasatinib_sensitive, non_tumor),
    }

    for metric_name, value in metrics.items():
        summary_rows.append({
            "Top_percent": top_percent,
            "Metric": metric_name,
            "Percent": value,
            f"{drugA_name}_threshold": lapatinib_threshold,
            f"{drugB_name}_threshold": dasatinib_threshold,
            "n_tumor_like": tumor_like.sum(),
            "n_non_tumor": non_tumor.sum(),
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(result_path("04_combination_coverage_metrics_threshold_summary.csv"), index=False)

print(summary_df)


In [ ]:
# =========================
# Multi-panel barplot
# =========================
metric_order = ['Lapatinib coverage in tumor-like',
 'Dasatinib coverage in tumor-like',
 'Combined coverage in tumor-like',
 'New tumor-like coverage by Dasatinib',
 'Non-tumor covered by Dasatinib']

short_metric_labels = {
    f"{drugA_name} coverage in tumor-like": f"{drugA_name}\ntumor-like",
    f"{drugB_name} coverage in tumor-like": f"{drugB_name}\ntumor-like",
    "Combined coverage in tumor-like": "Combined\ntumor-like",
    f"New tumor-like coverage by {drugB_name}": f"New by\n{drugB_name}",
    f"Non-tumor covered by {drugB_name}": f"{drugB_name}\nnon-tumor",
}

bar_colors = [
    "#F46D43",
    "#7FB069",
    "#7A3EB1",
    "#4C78A8",
    "#BDBDBD",
]

n = len(top_percents)
fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4.2), sharey=True)

if n == 1:
    axes = [axes]

global_ymax = min(100, summary_df["Percent"].max() * 1.25)

for ax, top_percent in zip(axes, top_percents):
    tmp = summary_df[
        summary_df["Top_percent"] == top_percent
    ].copy()

    tmp["Metric"] = pd.Categorical(
        tmp["Metric"],
        categories=metric_order,
        ordered=True
    )
    tmp = tmp.sort_values("Metric")

    labels = [short_metric_labels[m] for m in tmp["Metric"]]
    values = tmp["Percent"].values

    bars = ax.bar(
        labels,
        values,
        color=bar_colors,
        edgecolor="black",
        linewidth=0.8
    )

    ax.set_title(
        f"Top {top_percent}%",
        fontsize=12,
        weight="bold"
    )

    ax.set_ylim(0, global_ymax)
    ax.tick_params(axis="x", labelsize=8, rotation=0)
    ax.tick_params(axis="y", labelsize=9)

    for bar in bars:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + global_ymax * 0.025,
            f"{h:.1f}%",
            ha="center",
            va="bottom",
            fontsize=8
        )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0].set_ylabel("Coverage (%)", fontsize=12)

fig.suptitle(
    "Spatial therapeutic coverage across sensitivity thresholds",
    fontsize=15,
    weight="bold",
    y=1.04
)

plt.tight_layout()
plt.savefig(result_path("04_combination_coverage_metrics_multipanel_barplot.pdf"),
    dpi=600,
    bbox_inches="tight"
)
plt.savefig(result_path("04_combination_coverage_metrics_multipanel_barplot.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# =========================
# Basic settings.
# =========================
drugA_col = LAPATINIB_SCORE_COL
drugB_col = DASATINIB_SCORE_COL
drugA_name = LAPATINIB_DRUG_NAME
drugB_name = DASATINIB_DRUG_NAME
top_percent_A = 25
top_percent_B = 25

n_perm = 1000
random_seed = 43

# =========================
# Data preparation.
# =========================
adata_use = adata[
    adata.obs[CLASSIFICATION_COLUMN].notna()
    & adata.obs[drugA_col].notna()
    & adata.obs[drugB_col].notna()
].copy()

sA = adata_use.obs[drugA_col].astype(float).values
sB = adata_use.obs[drugB_col].astype(float).values
cls = adata_use.obs[CLASSIFICATION_COLUMN].astype(str).values

# Main analysis: Tumor + Invasive.
tumor_like = np.isin(cls, TUMOR_LIKE_CLASSES)
non_tumor = ~tumor_like

print("Number of spots used:", adata_use.n_obs)
print("Tumor-like spots:", tumor_like.sum())
print("Non-tumor spots:", non_tumor.sum())

# =========================
# Define sensitive regions.
# =========================
tA = np.nanpercentile(sA, 100 - top_percent_A)
tB = np.nanpercentile(sB, 100 - top_percent_B)

senA = sA >= tA
senB = sB >= tB

print(f"{drugA_name} threshold (top {top_percent_A}%): {tA:.4f}")
print(f"{drugB_name} threshold (top {top_percent_B}%): {tB:.4f}")
print(f"{drugA_name} sensitive spots:", senA.sum())
print(f"{drugB_name} sensitive spots:", senB.sum())


# Dasatinib additional coverage = B-sensitive but not A-sensitive.
added_by_B_obs = (~senA) & senB

observed_new_coverage = coverage_percent(added_by_B_obs, tumor_like)

print(f"Observed new tumor-like coverage by {drugB_name}: {observed_new_coverage:.2f}%")

# =========================
# permutation test
# =========================
rng = np.random.default_rng(random_seed)
perm_new_coverages = []

for i in range(n_perm):
    # Shuffle Dasatinib-sensitive labels while preserving the number of sensitive spots.
    senB_perm = rng.permutation(senB)

    added_by_B_perm = (~senA) & senB_perm
    perm_cov = coverage_percent(added_by_B_perm, tumor_like)
    perm_new_coverages.append(perm_cov)

perm_new_coverages = np.array(perm_new_coverages)

# One-sided p-value: probability that the null is at least the observed value.
p_perm = (np.sum(perm_new_coverages >= observed_new_coverage) + 1) / (n_perm + 1)

print(f"Permutation p-value: {p_perm:.4g}")
print(f"Null mean: {perm_new_coverages.mean():.2f}%")
print(f"Null std: {perm_new_coverages.std(ddof=1):.2f}%")

# =========================
# Save permutation values.
# =========================
perm_df = pd.DataFrame({
    "perm_new_tumor_like_coverage": perm_new_coverages
})
perm_df.to_csv(result_path("05_dasatinib_new_coverage_permutation_values.csv"), index=False)

summary_df = pd.DataFrame({
    "Metric": [
        "Observed new tumor-like coverage by Dasatinib",
        "Null mean",
        "Null std",
        "Permutation p-value",
        "n permutations",
        "Tumor-like spots",
        "Top percent A",
        "Top percent B"
    ],
    "Value": [
        observed_new_coverage,
        perm_new_coverages.mean(),
        perm_new_coverages.std(ddof=1),
        p_perm,
        n_perm,
        tumor_like.sum(),
        top_percent_A,
        top_percent_B
    ]
})
summary_df.to_csv(result_path("05_dasatinib_new_coverage_permutation_summary.csv"), index=False)

print(summary_df)

# =========================
# Plot histogram with the observed-value line.
# =========================
fig, ax = plt.subplots(figsize=(6.5, 4.8))

ax.hist(
    perm_new_coverages,
    bins=30,
    edgecolor="black",
    linewidth=0.8,
    alpha=0.85
)

# Red line: observed value.
ax.axvline(
    observed_new_coverage,
    color="red",
    linestyle="-",
    linewidth=2.5,
    label=f"Observed = {observed_new_coverage:.2f}%"
)

# Black dashed line: null mean.
ax.axvline(
    perm_new_coverages.mean(),
    color="black",
    linestyle="--",
    linewidth=1.8,
    label=f"Null mean = {perm_new_coverages.mean():.2f}%"
)

ax.set_xlabel(f"New tumor-like coverage by {drugB_name} (%)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title(
    f"Permutation test for additional tumor-like coverage\n"
    f"({drugA_name} vs {drugB_name}, top {top_percent_A}% / {top_percent_B}%)",
    fontsize=13,
    weight="bold"
)

# Annotate the p-value on the plot.
ax.text(
    0.98, 0.95,
    f"Observed = {observed_new_coverage:.2f}%\n"
    f"Null mean = {perm_new_coverages.mean():.2f}%\n"
    f"Permutation p = {p_perm:.4g}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=10,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="gray", alpha=0.9)
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig(result_path("05_dasatinib_new_coverage_permutation_hist.pdf"), dpi=600, bbox_inches="tight")
plt.savefig(result_path("05_dasatinib_new_coverage_permutation_hist.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# =========================
# Basic settings.
# =========================
drugA_col = LAPATINIB_SCORE_COL
drugB_col = DASATINIB_SCORE_COL
drugA_name = LAPATINIB_DRUG_NAME
drugB_name = DASATINIB_DRUG_NAME
top_percents = SENSITIVITY_TOP_PERCENTAGES
n_perm = 1000
random_seed = 43

# =========================
# Data preparation.
# =========================
adata_use = adata[
    adata.obs[CLASSIFICATION_COLUMN].notna()
    & adata.obs[drugA_col].notna()
    & adata.obs[drugB_col].notna()
].copy()

sA = adata_use.obs[drugA_col].astype(float).values
sB = adata_use.obs[drugB_col].astype(float).values
cls = adata_use.obs[CLASSIFICATION_COLUMN].astype(str).values

# Main-analysis definition: Tumor + Invasive.
tumor_like = np.isin(cls, TUMOR_LIKE_CLASSES)
non_tumor = ~tumor_like

print("Number of spots used:", adata_use.n_obs)
print("Tumor-like spots:", tumor_like.sum())
print("Non-tumor spots:", non_tumor.sum())

# To include Surrounding tumor, use:
# tumor_like = np.isin(cls, ["Tumor", "Invasive", "Surrounding tumor"])


# =========================
# Multi-threshold permutation test.
# =========================
rng = np.random.default_rng(random_seed)

summary_rows = []
perm_dict = {}   # Store the null distribution for each threshold for plotting

for top_percent in top_percents:
    # ---------- Define sensitive regions ----------
    tA = np.nanpercentile(sA, 100 - top_percent)
    tB = np.nanpercentile(sB, 100 - top_percent)

    senA = sA >= tA
    senB = sB >= tB

    # Observed additional coverage.
    added_by_B_obs = (~senA) & senB
    observed_new_cov = coverage_percent(added_by_B_obs, tumor_like)

    # Permutation: shuffle Dasatinib-sensitive labels.
    perm_vals = []
    for i in range(n_perm):
        senB_perm = rng.permutation(senB)
        added_by_B_perm = (~senA) & senB_perm
        perm_cov = coverage_percent(added_by_B_perm, tumor_like)
        perm_vals.append(perm_cov)

    perm_vals = np.array(perm_vals)
    perm_dict[top_percent] = perm_vals

    null_mean = perm_vals.mean()
    null_std = perm_vals.std(ddof=1)

    # One-sided p-value: probability that the null is at least the observed value.
    p_perm = (np.sum(perm_vals >= observed_new_cov) + 1) / (n_perm + 1)

    # z-score for display.
    if null_std > 0:
        z_score = (observed_new_cov - null_mean) / null_std
    else:
        z_score = np.nan

    # Also record Dasatinib non-tumor coverage.
    non_tumor_cov_B = coverage_percent(senB, non_tumor)

    summary_rows.append({
        "Top_percent": top_percent,
        f"{drugA_name}_threshold": tA,
        f"{drugB_name}_threshold": tB,
        f"{drugA_name}_sensitive_n": int(senA.sum()),
        f"{drugB_name}_sensitive_n": int(senB.sum()),
        "Observed_new_tumor_like_coverage": observed_new_cov,
        "Null_mean": null_mean,
        "Null_std": null_std,
        "Permutation_p_value": p_perm,
        "Z_score": z_score,
        f"{drugB_name}_non_tumor_coverage": non_tumor_cov_B,
        "Tumor_like_n": int(tumor_like.sum()),
        "Non_tumor_n": int(non_tumor.sum()),
        "n_permutations": n_perm
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(result_path("05_dasatinib_multithreshold_permutation_summary.csv"), index=False)

print(summary_df)

# Save all permutation values in long-table format.
perm_long = []
for top_percent, vals in perm_dict.items():
    for v in vals:
        perm_long.append({
            "Top_percent": top_percent,
            "Permuted_new_tumor_like_coverage": v
        })
perm_long_df = pd.DataFrame(perm_long)
perm_long_df.to_csv(result_path("05_dasatinib_multithreshold_permutation_values_long.csv"), index=False)

print("\nSaved:")
print(" - 05_dasatinib_multithreshold_permutation_summary.csv")
print(" - 05_dasatinib_multithreshold_permutation_values_long.csv")


In [ ]:
# =========================
# Multi-panel histograms
# =========================
n = len(top_percents)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), sharey=True)

if n == 1:
    axes = [axes]

global_xmin = min([perm_dict[t].min() for t in top_percents] + [summary_df["Observed_new_tumor_like_coverage"].min()])
global_xmax = max([perm_dict[t].max() for t in top_percents] + [summary_df["Observed_new_tumor_like_coverage"].max()])

for ax, top_percent in zip(axes, top_percents):
    perm_vals = perm_dict[top_percent]
    row = summary_df[summary_df["Top_percent"] == top_percent].iloc[0]

    observed = row["Observed_new_tumor_like_coverage"]
    null_mean = row["Null_mean"]
    p_perm = row["Permutation_p_value"]

    ax.hist(
        perm_vals,
        bins=25,
        edgecolor="black",
        linewidth=0.8,
        alpha=0.85
    )

    ax.axvline(
        observed,
        color="red",
        linewidth=2.5,
        label="Observed"
    )

    ax.axvline(
        null_mean,
        color="black",
        linestyle="--",
        linewidth=1.8,
        label="Null mean"
    )

    ax.set_title(f"Top {top_percent}%", fontsize=12, weight="bold")
    ax.set_xlim(global_xmin, global_xmax)

    if p_perm < 0.001:
        ptxt = "p<0.001"
    else:
        ptxt = f"p={p_perm:.3f}"

    ax.text(
        0.97, 0.95,
        f"Obs = {observed:.1f}%\nMean = {null_mean:.1f}%\n{ptxt}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=8.5,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="gray", alpha=0.9)
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=9)

axes[0].set_ylabel("Count", fontsize=12)

# Global x-axis label: place it above the legend.
fig.supxlabel(
    f"New tumor-like coverage by {drugB_name} (%)",
    fontsize=12,
    y=0.105
)

# Main title: move slightly upward.
fig.suptitle(
    f"Permutation distributions across thresholds\n"
    f"({drugA_name} vs {drugB_name})",
    fontsize=15,
    weight="bold",
    y=0.99
)

# Shared legend: place at the bottom.
handles = [
    plt.Line2D([0], [0], color="red", linewidth=2.5, label="Observed"),
    plt.Line2D([0], [0], color="black", linestyle="--", linewidth=1.8, label="Null mean")
]

fig.legend(
    handles=handles,
    frameon=False,
    fontsize=10,
    loc="lower center",
    ncol=2,
    bbox_to_anchor=(0.5, 0.015)
)

# Manually leave vertical space.
plt.subplots_adjust(
    left=0.04,
    right=0.99,
    top=0.78,
    bottom=0.20,
    wspace=0.25
)

plt.savefig(result_path("05_dasatinib_multithreshold_permutation_histograms.pdf"),
    dpi=600,
    bbox_inches="tight"
)
plt.savefig(result_path("05_dasatinib_multithreshold_permutation_histograms.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
gdsc2_file = DATA_DIR / "GDSC2_fitted_dose_response_27Oct23.xlsx"

gdsc_df = pd.read_excel(gdsc2_file)
gdsc_df["Dataset"] = "GDSC2"

print("GDSC2 shape:", gdsc_df.shape)
print(gdsc_df.columns.tolist())
print(gdsc_df.head())

key_columns = ["CELL_LINE_NAME", "COSMIC_ID", "TCGA_DESC", "DRUG_NAME", "LN_IC50", "AUC", "Z_SCORE"]
for column_name in key_columns:
    if column_name in gdsc_df.columns:
        print(column_name, "exists")
    else:
        print(column_name, "NOT FOUND")


In [ ]:
gdsc_drug_name = DASATINIB_DRUG_NAME

dasatinib_records_df = gdsc_df[
    gdsc_df["DRUG_NAME"].astype(str).str.lower() == gdsc_drug_name.lower()
].copy()

print("Dasatinib records:", dasatinib_records_df.shape)
print(dasatinib_records_df[["Dataset", "CELL_LINE_NAME", "TCGA_DESC", "DRUG_NAME", "LN_IC50", "AUC", "Z_SCORE"]].head())

dasatinib_breast_df = dasatinib_records_df[
    dasatinib_records_df["TCGA_DESC"].astype(str).str.upper().eq("BRCA")
].copy()
dasatinib_breast_df = dasatinib_breast_df.dropna(subset=["LN_IC50"])

print("Dasatinib breast records:", dasatinib_breast_df.shape)
print(dasatinib_breast_df[["Dataset", "CELL_LINE_NAME", "TCGA_DESC", "LN_IC50", "AUC", "Z_SCORE"]].head())

# Lower LN_IC50 indicates higher sensitivity, so the sign is inverted.
dasatinib_breast_df["Dasatinib_sensitivity_like"] = -dasatinib_breast_df["LN_IC50"]
dasatinib_breast_df = dasatinib_breast_df.sort_values("Dasatinib_sensitivity_like", ascending=False)
dasatinib_breast_df.to_csv(result_path("external_GDSC_Dasatinib_breast_cell_lines.csv"), index=False)

print(dasatinib_breast_df[[
    "Dataset", "CELL_LINE_NAME", "LN_IC50", "AUC", "Z_SCORE", "Dasatinib_sensitivity_like"
]].head(20))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

ax.bar(
    dasatinib_breast_df["CELL_LINE_NAME"].astype(str),
    dasatinib_breast_df["Dasatinib_sensitivity_like"],
    color="#7FB069",
    edgecolor="black",
    linewidth=0.5,
)

median_sensitivity_like = dasatinib_breast_df["Dasatinib_sensitivity_like"].median()
ax.axhline(
    median_sensitivity_like,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Median",
)

ax.set_ylabel("Dasatinib sensitivity-like score (-LN_IC50)", fontsize=12)
ax.set_xlabel("Breast cancer cell lines", fontsize=12)
ax.set_title(
    "External GDSC2 response of Dasatinib in breast cancer cell lines",
    fontsize=13,
    weight="bold",
)
ax.tick_params(axis="x", rotation=90, labelsize=7)
ax.tick_params(axis="y", labelsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(False)
ax.legend(frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig(result_path("external_GDSC_Dasatinib_breast_cell_lines_barplot_short.pdf"), dpi=600, bbox_inches="tight")
plt.savefig(result_path("external_GDSC_Dasatinib_breast_cell_lines_barplot_short.png"), dpi=300, bbox_inches="tight")
plt.show()
